# 04 기권 기준과 실패 분석

목표: dev로 설정을 선택한 뒤 고정 test의 실패를 유형·말투별로 분석합니다.

**준비:** 저장소 터미널에서 python -m pip install -r requirements/week11-offline.txt 실행.
VS Code 커널에서 저장소 .venv의 **Python 3.11** 선택 후 위에서 아래로 실행합니다.
각 파일은 다른 Notebook의 변수를 요구하지 않습니다. GPU·Ollama·API 키는 필요 없습니다.

**데이터:** 교수 작성 가상 쇼핑 정책입니다. 실제 기업 규정이나 고객 기록이 아닙니다.
핵심 단계는 수업에서 함께 실행하고 확장 실험은 복습용입니다. [실습 안내](../README.md).


In [ ]:
from pathlib import Path
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week11/lab/evaluation.py").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("저장소 안에서 Notebook을 실행하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("저장소 .venv의 Python 3.11 커널을 선택하세요.")
import pandas as pd
from IPython.display import display
from week11.lab.evaluation import load_data, Config, evaluate, summarize, select_on_dev

docs, questions = load_data()
dev = [q for q in questions if q["split"] == "dev"]
test = [q for q in questions if q["split"] == "test"]
print("환경 준비 완료", sys.version.split()[0], "문서", len(docs), "질문", len(questions))

## 1. 설정 선택

교육용 utility = 근거 확보율 − 답변 불가 수락률 − 0.25×답변 가능 기권율.

In [ ]:
selected, trials = select_on_dev(docs, dev)
print(selected)
display(
    pd.DataFrame([{**t["config"], "utility": t["utility"], **t["summary"]} for t in trials])
    .sort_values("utility", ascending=False)
    .head(8)
)

**확인과 해석**

모든 사업의 공통 목표가 아닙니다. 동점은 작은 k, 높은 threshold 순으로 해소합니다.

## 2. 임계값 양면

기권 증가로 위험한 시도와 유용한 응답이 함께 줄 수 있습니다.

In [ ]:
from dataclasses import replace

curve = [
    {"threshold": t, **summarize(evaluate(docs, dev, replace(selected, threshold=t)))}
    for t in [0, 0.15, 0.3, 0.45, 0.6]
]
display(
    pd.DataFrame(curve)[
        ["threshold", "coverage", "false_accept", "missed_answer", "evidence_complete"]
    ]
)

**확인과 해석**

cosine 0.3은 정답 확률 30%가 아닙니다.

## 3. 고정 test

이후 test에 맞춰 조정하면 새 평가 자료가 필요합니다.

In [ ]:
baseline = evaluate(docs, test, Config("word-all"))
candidate = evaluate(docs, test, selected)
display(pd.DataFrame({"baseline": summarize(baseline), "candidate": summarize(candidate)}))

**확인과 해석**

Recall과 필수 근거 확보율의 차이를 기권 정책과 연결해 봅니다.

## 4. 말투별 분석

전체 평균에 가려진 취약 유형을 찾습니다.

In [ ]:
frame = pd.DataFrame(candidate)
display(
    pd.DataFrame(
        [
            {"style": style, **summarize(group.to_dict("records"))}
            for style, group in frame.groupby("style")
        ]
    )
)
display(pd.crosstab(frame["slice"], frame["error"]))

**확인과 해석**

작은 유형은 순위 확정보다 후속 조사 단서로 활용합니다.

## 5. 실패 근거 추적

질문·정답·검색 결과·정책 판단을 연결합니다.

In [ ]:
failures = [r for r in candidate if r["error"] not in ["evidence_complete", "correct_abstain"]]
for r in failures[:3]:
    print(r["id"], r["query"], r["error"], "필수", r["gold_ids"])
    display(pd.DataFrame(r["ranked"])[["id", "title", "score"]] if r["ranked"] else pd.DataFrame())

**확인과 해석**

검색 실패를 무조건 프롬프트 수정으로 해결하려 하지 않습니다.

## 6. 분리 규칙 강제

도구가 실수를 먼저 차단합니다.

In [ ]:
try:
    select_on_dev(docs, test)
except ValueError as error:
    print("test 선택 금지:", error)

**확인과 해석**

이미 본 test에 맞춰 새 후보를 만드는 사람의 행동까지 막아주지는 못합니다.

## 확장 실험과 프로젝트 적용

1. 가장 비용이 큰 실패 두 개의 사용자 영향을 적습니다.
2. 동의어·문서 개정·추가 질문 중 하나만 dev에서 실험합니다.
3. 새 의도로 개선을 확인합니다.

**실행 실패 시:** ModuleNotFoundError는 커널과 설치 터미널이 같은 .venv인지 확인합니다.
ROOT 오류는 저장소 안에서 파일을 열면 해결됩니다. 변수가 섞였다면 Restart Kernel 후 Run All.
예외 실험은 try/except로 설명하므로 정상 실행에서는 빨간 오류로 끝나지 않습니다.

**기록:** 변경 설정 / 예상 / 실제 관찰 / 실패 질문 ID / 다음 행동을 5줄로 적습니다.
개인 프로젝트 원문·연락처·API 키는 공개 GitHub에 올리지 않습니다.
